# <p style="text-align:center;"> **Task 1**</p>

In [14]:
import urllib.request
import time
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

## **Web Scraping and Parsing**

In [15]:
# Retail Directory
BASE_DIR = "http://mlg.ucd.ie/modules/python/sources/retail/"
YEAR = 2025

# Num pages per Quarter
PAGES_PER_QUARTER = {
    1: 16,
    2: 17,
    3: 17,
    4: 16
}

# Add delay between requests
DELAY = 0.1

### Helper and Data Cleaning Functions

In [ ]:
def get_html(url):
    """
    Gets the html for a page and returns a string.
    """
    response = urllib.request.urlopen(url)
    html = response.read().decode()
    return html

#=====================================================================
def clean_text(text):
    """
    Removes whitespaces and non breaking spaces.
    """
    return text.replace("\xa0", " ").strip()

#========================================================================
def parse_money(text):
    """
    Converts money strings to floats 

    Examples covered:
      '€ 7.50'  -> 7.50
      'EUR 35.00' -> 35.00
      '18.50' -> 18.50
    """
    t = clean_text(text)
    t = t.replace("€", "")
    t = t.replace("EUR", "")
    t = t.replace(",", "")
    t = t.strip()

    return float(t)

#===========================================================================
def parse_quantity(text):
    """
    Converts quantities to integers.
    Assuming -1 means a refund and not wrong value
    """
    t = clean_text(text)
    
    return int(t)

#=============================================================================
def parse_date(text):
    """
    Converts the date formats to YYYY-MM-DD format.

    Formats covered:
    2025-10-01  
    01 Oct 2025  
    01-10-2025   
    Oct 01, 2025 
    """
    t = clean_text(text)

    formats = ["%Y-%m-%d", "%d %b %Y", "%d-%m-%Y", "%b %d, %Y"]

    for fmt in formats:
        try:
            return datetime.strptime(t, fmt).date().isoformat()
        except:
            pass

    # If none of the formats worked
    return None 

#============================================================================
def parse_customer_details(details_text):
    """
    Extracts customer_id, location, gender, and age_band from the "Customer Details" section

    The labels are inconsistent on the site, so we check common variants like:
      'Customer ID: 2487' or 'Cust ID: 10219' or 'ID: 15687'
      'Location Galway' or 'City: Cork' or 'From: Offaly'
      'Age Range: 55-64' or 'Age Group: 35-44' or 'Age: 45-54'
    """
    cust_details = {
        "customer_id": None,
        "location": None,
        "gender": None,
        "age_band": None
    }
    # Handle both <br> tags and \n
    details_text = details_text.replace("<br>", "\n").replace("<br/>", "\n").replace("<br />", "\n")
       
    # split into lines 
    lines = details_text.split("\n")

    for line in lines:
        line = clean_text(line)

        # ----- Customer ID -----
        # cover: "Customer ID: 2487" / "Cust ID: 10219" / "ID: 15687" / "ID 18141"
        if "ID" in line:
            # make it easier by removing known labels
            tmp = line.replace("Customer ID:", "").replace("Cust ID:", "").replace("ID:", "").replace("ID", "").strip()
            # now tmp might be just a number
            try:
                cust_details["customer_id"] = int(tmp)
            except:
                pass

        # Location
        if line.startswith("Location"):
            cust_details["location"] = clean_text(line.replace("Location", "").replace(":", ""))
        if line.startswith("City"):
            cust_details["location"] = clean_text(line.replace("City", "").replace(":", ""))
        if line.startswith("From"):
            cust_details["location"] = clean_text(line.replace("From", "").replace(":", ""))

        #Gender
        if line.startswith("Gender"):
            cust_details["gender"] = clean_text(line.replace("Gender", "").replace(":", ""))

        # Age band
        if line.startswith("Age Range"):
            cust_details["age_band"] = clean_text(line.replace("Age Range", "").replace(":", ""))
        if line.startswith("Age Group"):
            cust_details["age_band"] = clean_text(line.replace("Age Group", "").replace(":", ""))
        if line.startswith("Age Category"):
            cust_details["age_band"] = clean_text(line.replace("Age Category", "").replace(":", ""))
        if line.startswith("Age"):
            cust_details["age_band"] = clean_text(line.replace("Age", "").replace(":", ""))

    return cust_details


### Parse Page/Table Functions


In [17]:
def parse_transaction_info(table):
    """
    Each transaction is a small table with rows like:
    Product, Sale Date, Category, Quantity, Total Price, Total Profit, Payment Type, Customer Details

    This function converts that table to a python dict
    """
    row = {
        "product": None,
        "sale_date": None,
        "category": None,
        "quantity": None,
        "total_price": None,
        "total_profit": None,
        "payment_type": None,
        "customer_id": None,
        "location": None,
        "gender": None,
        "age_band": None
    }

    # Each row is a <tr> with two <td> cells label and value
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 2:
            continue

        label = clean_text(tds[0].get_text())
        value = tds[1].get_text("\n")   # keep line breaks so customer details split 
        value = clean_text(value)

        # Match based on label text
        if label.startswith("Product"):
            row["product"] = clean_text(tds[1].get_text(" "))
        elif label.startswith("Sale Date"):
            row["sale_date"] = parse_date(value)
        elif label.startswith("Category"):
            row["category"] = value
        elif label.startswith("Quantity"):
            row["quantity"] = parse_quantity(value)
        elif label.startswith("Total Price"):
            row["total_price"] = parse_money(value)
        elif label.startswith("Total Profit"):
            row["total_profit"] = parse_money(value)
        elif label.startswith("Payment Type"):
            row["payment_type"] = value
        elif label.startswith("Customer Details"):
            cust = parse_customer_details(value)
            row["customer_id"] = cust["customer_id"]
            row["location"] = cust["location"]
            row["gender"] = cust["gender"]
            row["age_band"] = cust["age_band"]

    return row


In [18]:
def parse_page(html):
    """
    Each transaction is a <table class="transaction-info">.
    This finds all those tables and converts them into a list of dictionaries.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Select every transaction table on the page
    tables = soup.select("table.transaction-info")

    rows = []
    for table in tables:
        rows.append(parse_transaction_info(table))

    return rows


## **Main Data Scraping Loop**

In [ ]:
all_rows = []

for quarter, max_page in PAGES_PER_QUARTER.items():
    for page in range(1, max_page + 1):

        url = f"{BASE_DIR}{YEAR}-Q{quarter}-page{page:02d}.html"

        html = get_html(url)
        page_rows = parse_page(html)

        # add metadata columns so we know where data came from
        for r in page_rows:
            r["year"] = YEAR
            r["quarter"] = quarter
            r["page"] = page

        all_rows.extend(page_rows)
        time.sleep(DELAY)

retail_raw = pd.DataFrame(all_rows)

## **Save Data to CSV file**

In [ ]:
retail_raw.to_csv("retail_raw.csv", index=False, encoding="utf-8")

Saved: retail_raw_2025.csv
